# Semana 03: Reforço do GitFlow & Deploy Automatizado no Vercel

Este notebook reforça e consolida a aplicação prática do **GitFlow**, conectando o fluxo de controle de versão ao **Deploy Automatizado no Vercel** (Continuous Deployment inicial), demonstrando como ambientes de *Preview* e *Produção* são atualizados automaticamente a cada novidade na árvore do Git.

### Objetivos de aprendizagem
- Reforçar a mecânica do **GitFlow** integrando o fluxo de código ao ciclo de deploy contínuo.
- Compreender o funcionamento do **Deploy Automatizado** por Webhooks entre GitHub e Vercel.
- Entender o conceito e a utilidade dos **Preview Deployments** (para branches `develop` e `feature/*`) vs **Production Deployments** (ao mesclar em `main`).
- Dominar os comandos de automação via **Vercel CLI** e arquivos de configuração `vercel.json`.
- Executar um simulador em Python que roteia deploys do Vercel com base nas branches do GitFlow.

---


## 1. Fundamentação Teórica

### 1.1 Conectando o GitFlow à Automação de Deploy no Vercel

A grande vantagem de integrar o **GitFlow** com plataformas modernas de publicação (como o Vercel) é que a arquitetura de ambientes acompanha automaticamente o estado das branches:

![Deploy Automatizado no Vercel com GitFlow](img/vercel_gitflow_pipeline.jpg)

#### Mapeamento de Branches x Ambientes de Deploy:

1. **Branches `feature/*` (Preview Isolado por PR):**
   - Cada push na branch de funcionalidade dispara uma compilação automática no Vercel, gerando uma **URL de Preview única e efêmera** (ex: `https://smartn1-git-feature-oee.vercel.app`).
   - Permite que engenheiros, QAs e clientes validem o recurso sem afetar outros desenvolvedores.

2. **Branch `develop` (Ambiente de Staging/Homologação):**
   - Quando a Pull Request da feature é aprovada e mesclada na `develop`, o Vercel atualiza o **Ambiente de Homologação** (ex: `https://smartn1-dev.vercel.app`).

3. **Branch `main` (Ambiente de Produção Oficial):**
   - Ao concluir uma release (`release/v1.1.0`) ou hotfix (`hotfix/v1.0.1`) e realizar o merge na `main`, o Vercel atualiza o **Ambiente de Produção Oficial** (ex: `https://smartn1.vercel.app`) com **Zero Downtime** e troca instantânea de ponteiro DNS.

---

### 1.2 Mecânica de Webhooks e Inspecção de Build no Vercel

Quando um desenvolvedor executa `git push` no repositório remoto (GitHub), o fluxo de integração executa os seguintes passos automatizados:

```text
  [Desenvolvedor: git push origin feature/painel] 
                      |
                      v
  [GitHub Repository (Recebe o commit)] 
                      |
                      +--- (Envia Webhook HTTP POST JSON) ---> [Vercel Engine]
                                                                   |
                                                                   v
                                                       1. Instala dependências
                                                       2. Compila aplicação
                                                       3. Gera URL de Preview
                                                                   |
                                                                   v
                                                       [Injeta Status no GitHub PR]
```

---

### 1.3 Opcional: Gerenciamento via Vercel CLI (`vercel`)

Para ambientes de linha de comando ou automação por scripts, a **Vercel CLI** permite realizar a implantação diretamente do terminal:

```bash
# 1. Instalação da CLI
npm install -g vercel

# 2. Deploy em Ambiente de Preview (Desenvolvimento/Staging)
vercel

# 3. Deploy Oficial em Produção
vercel --prod
```

---


## 2. Prática — Simulador do Engine de Deploy Vercel em Python

Nesta atividade prática, desenvolveremos um script Python que simula o webhook enviado pelo GitHub ao Vercel, inspecionando o evento, determinando o tipo de deploy gerado (*Preview* ou *Produção*), montando a URL de publicação e simulando a checagem de status no Pull Request.

In [2]:
import datetime
import json

# Eventos de commit e PR simulados no GitHub para a plataforma da fábrica
eventos_webhook_github = [
    {"commit_sha": "a8f9e01", "ref": "refs/heads/feature/grafico-oee", "autor": "joao.silva", "pr_id": 14},
    {"commit_sha": "b2c3d4e", "ref": "refs/heads/develop", "autor": "tech.lead", "pr_id": None},
    {"commit_sha": "f5e6d7c", "ref": "refs/heads/release/v1.2.0", "autor": "qa.team", "pr_id": 18},
    {"commit_sha": "9a8b7c6", "ref": "refs/heads/main", "autor": "devops.bot", "pr_id": None}
]

def processar_webhook_vercel(eventos):
    relatorio_deploys = []
    
    for ev in eventos:
        ref = ev["ref"]
        sha = ev["commit_sha"]
        branch_nome = ref.replace("refs/heads/", "")
        
        if branch_nome == "main":
            categoria = "PRODUCTION DEPLOYMENT (AO VIVO)"
            url_final = "https://smartn1-fabrica.vercel.app"
            status_pr = "PROMOVIDO PARA PRODUÇÃO OFICIAL"
        elif branch_nome == "develop":
            categoria = "STAGING DEPLOYMENT (HOMOLOGAÇÃO)"
            url_final = "https://smartn1-dev.vercel.app"
            status_pr = "AMBIENTE DE INTEGRAÇÃO ATUALIZADO"
        else:
            categoria = "PREVIEW DEPLOYMENT (TESTE ISOLADO)"
            clean_branch = branch_nome.replace("/", "-")
            url_final = f"https://smartn1-git-{clean_branch}-{sha}.vercel.app"
            status_pr = f"PREVIEW READY (PR #{ev['pr_id'] or 'Local'})"
            
        relatorio_deploys.append({
            "hora": datetime.datetime.now().strftime("%H:%M:%S"),
            "branch": branch_nome,
            "commit": sha,
            "categoria": categoria,
            "url": url_final,
            "status_github": status_pr
        })
        
    return relatorio_deploys

deploys = processar_webhook_vercel(eventos_webhook_github)

print("=========================================================")
print("PROCESSADOR DE DEPLOYS AUTOMÁTICOS VERCEL + GITFLOW")
print("=========================================================\n")
for d in deploys:
    print(f"[{d['hora']}] Branch: '{d['branch']}' | Commit: {d['commit']}")
    print(f"  - Categoria:     {d['categoria']}")
    print(f"  - URL Publicada: {d['url']}")
    print(f"  - GitHub Status: {d['status_github']}\n")


PROCESSADOR DE DEPLOYS AUTOMÁTICOS VERCEL + GITFLOW

[18:09:31] Branch: 'feature/grafico-oee' | Commit: a8f9e01
  - Categoria:     PREVIEW DEPLOYMENT (TESTE ISOLADO)
  - URL Publicada: https://smartn1-git-feature-grafico-oee-a8f9e01.vercel.app
  - GitHub Status: PREVIEW READY (PR #14)

[18:09:31] Branch: 'develop' | Commit: b2c3d4e
  - Categoria:     STAGING DEPLOYMENT (HOMOLOGAÇÃO)
  - URL Publicada: https://smartn1-dev.vercel.app
  - GitHub Status: AMBIENTE DE INTEGRAÇÃO ATUALIZADO

[18:09:31] Branch: 'release/v1.2.0' | Commit: f5e6d7c
  - Categoria:     PREVIEW DEPLOYMENT (TESTE ISOLADO)
  - URL Publicada: https://smartn1-git-release-v1.2.0-f5e6d7c.vercel.app
  - GitHub Status: PREVIEW READY (PR #18)

[18:09:31] Branch: 'main' | Commit: 9a8b7c6
  - Categoria:     PRODUCTION DEPLOYMENT (AO VIVO)
  - URL Publicada: https://smartn1-fabrica.vercel.app
  - GitHub Status: PROMOVIDO PARA PRODUÇÃO OFICIAL



---

## 3. Exercícios de Fixação e Avaliação

### Questão 1
Explique como o uso de **Preview Deployments** automatizados no Vercel para branches de funcionalidade (`feature/*`) evita que bugs sejam introduzidos na branch de integração `develop` antes da revisão de código.

### Questão 2
No contexto da integração GitFlow + Vercel, qual a diferença entre publicar uma alteração na branch `develop` em comparação com publicar uma alteração na branch `main`?

### Questão 3
Descreva o papel dos Webhooks no ecossistema GitHub/Vercel. O que acontece quando o GitHub notifica a plataforma sobre um novo commit em um Pull Request aberto?

### Questão 4
Suponha que uma falha visual seja identificada na URL de Preview de um Pull Request `feature/novo-menu`. Quais passos o desenvolvedor deve seguir no GitFlow local para corrigir o problema e atualizar a Preview URL no Vercel de forma automática?
